# Pronóstico con una red neuronal

Abordajes diarios de tren en Chicago (CTA). El cuaderno tiene seis partes:

1. Carga de datos
2. Series a tensores
3. Dataset con ventanas y DataLoaders
4. Entrenamiento de una red simple
5. Entrenamiento de una RNN Superficial
6. Entrenamiento de una RNN Profunda

## 1. Carga de datos

Leemos el CSV de abordajes diarios, acortamos los nombres de las columnas y usamos la fecha como índice. Quitamos `total` (es la suma de bus y tren) y las filas duplicadas.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [2]:
path = Path('CTA_-_Ridership_-_Daily_Boarding_Totals.csv')
df = pd.read_csv(path, parse_dates=["service_date"], thousands=",")
df.columns = ["date", "day_type", "bus", "rail", "total"]  # nombres cortos
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)  # total = bus + tren
df = df.drop_duplicates()  # meses duplicados: 2011-10 y 2014-07

## 2. Series a tensores

La serie de tren pasa a tensores `float32`, en millones de abordajes. Entrenamiento: 2016 a 2018. Validación: enero a mayo de 2019. Prueba: desde junio de 2019.

In [3]:
rail_train = torch.tensor(df["rail"]["2016-01-01":"2018-12-31"].values/1e6, dtype=torch.float32)
rail_valid = torch.tensor(df["rail"]["2019-01-01":"2019-05-31"].values/1e6, dtype=torch.float32)    
rail_test = torch.tensor(df["rail"]["2019-06-01":].values/1e6, dtype=torch.float32)

## 3. Dataset con ventanas y DataLoaders

`TimeSeriesDataset` arma pares para el pronóstico: una ventana de 56 días y el valor del día siguiente. Los DataLoaders entregan esos pares en lotes de 32. El de entrenamiento se baraja; validación y prueba conservan el orden.

In [4]:
class TimeSeriesDataset(torch.utils.data.Dataset):
    def __init__(self, series, window_length):
        self.series = series
        self.window_length = window_length

    def __len__(self):
        return len(self.series) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        end = idx + self.window_length  # 1st index after window
        window = self.series[idx : end]
        target = self.series[end]
        return window, target

In [5]:
dataset_train = TimeSeriesDataset(rail_train, window_length=56)
train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=32, shuffle=True)
dataset_valid = TimeSeriesDataset(rail_valid, window_length=56)
valid_loader = torch.utils.data.DataLoader(dataset_valid, batch_size=32, shuffle=False)
dataset_test = TimeSeriesDataset(rail_test, window_length=56)
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=32, shuffle = False)

## 4. Entrenamiento de una red simple

Una red fully connected lee la ventana de 56 días y predice el abordaje del día siguiente. Se entrena con pérdida de Huber y Adam. Cada 10 épocas se mide el MAE en validación y se escala de nuevo a abordajes, porque la serie está en millones.

In [6]:
import torch.nn as nn
import torchmetrics

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(56, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
).to(device)

loss_fn = nn.HuberLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
MAPE = torchmetrics.MeanAbsolutePercentageError().to(device)
n_epochs = 100

for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    MAPE.reset()
    for batch in train_loader:
        window, target = batch
        window, target = window.to(device), target.to(device)
        pred = model(window).squeeze(-1)
        loss = loss_fn(pred, target)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            for window, target in valid_loader:
                window = window.to(device)
                target = target.to(device)
                pred = model(window).squeeze(-1)
                MAPE.update(pred, target)
        mean_loss = total_loss / len(train_loader)
        mape = MAPE.compute()
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {mean_loss:.4f}, MAPE: {mape:.4f}")


Epoch 1/100, Loss: 0.0197, MAPE: 0.2853
Epoch 11/100, Loss: 0.0031, MAPE: 0.0803
Epoch 21/100, Loss: 0.0027, MAPE: 0.0793
Epoch 31/100, Loss: 0.0026, MAPE: 0.0813
Epoch 41/100, Loss: 0.0027, MAPE: 0.0759
Epoch 51/100, Loss: 0.0024, MAPE: 0.0770
Epoch 61/100, Loss: 0.0023, MAPE: 0.0785
Epoch 71/100, Loss: 0.0022, MAPE: 0.0811
Epoch 81/100, Loss: 0.0024, MAPE: 0.0881
Epoch 91/100, Loss: 0.0022, MAPE: 0.0902


## 5. Entrenamiento de una red neuronal recurrente superficial

La red recorre la ventana día a día. En cada paso concatena el abordaje con el estado oculto anterior, aplica una capa lineal y una tanh, y actualiza la memoria. La predicción sale del estado final. Se entrena con pérdida de Huber, Adam (`lr=0.0001`) y 1000 épocas. Cada 50 épocas se mide el MAPE en validación.

In [7]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.memory = nn.Sequential(
            nn.Linear(input_size + hidden_size, hidden_size),
            nn.Tanh()
        )
        self.output = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        batch_size, window_length, dimensionality = x.shape
        x_time_first = x.transpose(0, 1)
        H = torch.zeros(batch_size, self.hidden_size, device=x.device)
        for x_t in x_time_first:
            H = self.memory(torch.cat([x_t, H], dim=1))
        return self.output(H)

model = SimpleRNN(input_size=1, hidden_size=32, output_size=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [8]:
n_epochs = 1000
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    MAPE.reset()
    for batch in train_loader:
        window, target = batch
        window, target = window.to(device), target.to(device)
        window = window.unsqueeze(-1)  # (batch, 56) -> (batch, 56, 1)
        pred = model(window).squeeze(-1)
        loss = loss_fn(pred, target)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    if epoch % 50 == 0:
        model.eval()
        with torch.no_grad():
            for window, target in valid_loader:
                window = window.unsqueeze(-1).to(device)
                target = target.to(device)
                pred = model(window).squeeze(-1)
                MAPE.update(pred, target)
        mean_loss = total_loss / len(train_loader)
        mape = MAPE.compute()
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {mean_loss:.4f}, MAPE: {mape:.4f}")

Epoch 1/1000, Loss: 0.3100, MAPE: 1.1942
Epoch 51/1000, Loss: 0.0152, MAPE: 0.3503
Epoch 101/1000, Loss: 0.0131, MAPE: 0.3126
Epoch 151/1000, Loss: 0.0071, MAPE: 0.2016
Epoch 201/1000, Loss: 0.0041, MAPE: 0.1399
Epoch 251/1000, Loss: 0.0032, MAPE: 0.0923
Epoch 301/1000, Loss: 0.0030, MAPE: 0.1005
Epoch 351/1000, Loss: 0.0031, MAPE: 0.0884
Epoch 401/1000, Loss: 0.0029, MAPE: 0.0909
Epoch 451/1000, Loss: 0.0030, MAPE: 0.0863
Epoch 501/1000, Loss: 0.0027, MAPE: 0.0858
Epoch 551/1000, Loss: 0.0026, MAPE: 0.0863
Epoch 601/1000, Loss: 0.0025, MAPE: 0.0885
Epoch 651/1000, Loss: 0.0025, MAPE: 0.0871
Epoch 701/1000, Loss: 0.0025, MAPE: 0.0804
Epoch 751/1000, Loss: 0.0024, MAPE: 0.0819
Epoch 801/1000, Loss: 0.0024, MAPE: 0.0753
Epoch 851/1000, Loss: 0.0023, MAPE: 0.0803
Epoch 901/1000, Loss: 0.0023, MAPE: 0.0750
Epoch 951/1000, Loss: 0.0023, MAPE: 0.0781


### La misma red con `nn.RNN`

`nn.RNN` hace esa recurrencia. La predicción toma el estado del último día y lo pasa por una capa lineal. El entrenamiento no cambia.

In [9]:
class SimpleAbstractRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        output, hidden = self.rnn(x)
        return self.out(output[:, -1])

model = SimpleAbstractRNN(input_size=1, hidden_size=32, output_size=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [10]:
n_epochs = 1000
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    MAPE.reset()
    for batch in train_loader:
        window, target = batch
        window, target = window.to(device), target.to(device)
        window = window.unsqueeze(-1)  # (batch, 56) -> (batch, 56, 1)
        pred = model(window).squeeze(-1)
        loss = loss_fn(pred, target)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    if epoch % 50 == 0:
        model.eval()
        with torch.no_grad():
            for window, target in valid_loader:
                window = window.unsqueeze(-1).to(device)
                target = target.to(device)
                pred = model(window).squeeze(-1)
                MAPE.update(pred, target)
        mean_loss = total_loss / len(train_loader)
        mape = MAPE.compute()
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {mean_loss:.4f}, MAPE: {mape:.4f}")

Epoch 1/1000, Loss: 0.1049, MAPE: 0.5122
Epoch 51/1000, Loss: 0.0078, MAPE: 0.2105
Epoch 101/1000, Loss: 0.0043, MAPE: 0.1339
Epoch 151/1000, Loss: 0.0031, MAPE: 0.0894
Epoch 201/1000, Loss: 0.0028, MAPE: 0.0812
Epoch 251/1000, Loss: 0.0028, MAPE: 0.0826
Epoch 301/1000, Loss: 0.0026, MAPE: 0.0788
Epoch 351/1000, Loss: 0.0026, MAPE: 0.0751
Epoch 401/1000, Loss: 0.0026, MAPE: 0.0771
Epoch 451/1000, Loss: 0.0024, MAPE: 0.0735
Epoch 501/1000, Loss: 0.0024, MAPE: 0.0796
Epoch 551/1000, Loss: 0.0023, MAPE: 0.0746
Epoch 601/1000, Loss: 0.0023, MAPE: 0.0753
Epoch 651/1000, Loss: 0.0022, MAPE: 0.0843
Epoch 701/1000, Loss: 0.0023, MAPE: 0.0738
Epoch 751/1000, Loss: 0.0023, MAPE: 0.0746
Epoch 801/1000, Loss: 0.0023, MAPE: 0.0773
Epoch 851/1000, Loss: 0.0022, MAPE: 0.0765
Epoch 901/1000, Loss: 0.0022, MAPE: 0.0746
Epoch 951/1000, Loss: 0.0022, MAPE: 0.0772


## 6. Entrenamiento de una red neuronal recurrente profunda

Tres capas recurrentes apiladas (`num_layers=3`). Cada una recibe la secuencia de estados de la capa anterior. La predicción sigue saliendo del último paso de la capa de arriba, con el mismo entrenamiento.

In [11]:
class SimpleAbstractDeepRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=3, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        output, hidden = self.rnn(x)
        return self.out(output[:, -1])

model = SimpleAbstractDeepRNN(input_size=1, hidden_size=32, output_size=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [12]:
n_epochs = 1000
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    MAPE.reset()
    for batch in train_loader:
        window, target = batch
        window, target = window.to(device), target.to(device)
        window = window.unsqueeze(-1)  # (batch, 56) -> (batch, 56, 1)
        pred = model(window).squeeze(-1)
        loss = loss_fn(pred, target)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    if epoch % 50 == 0:
        model.eval()
        with torch.no_grad():
            for window, target in valid_loader:
                window = window.unsqueeze(-1).to(device)
                target = target.to(device)
                pred = model(window).squeeze(-1)
                MAPE.update(pred, target)
        mean_loss = total_loss / len(train_loader)
        mape = MAPE.compute()
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {mean_loss:.4f}, MAPE: {mape:.4f}")

Epoch 1/1000, Loss: 0.2376, MAPE: 0.7887
Epoch 51/1000, Loss: 0.0051, MAPE: 0.1445
Epoch 101/1000, Loss: 0.0029, MAPE: 0.0772
Epoch 151/1000, Loss: 0.0025, MAPE: 0.0686
Epoch 201/1000, Loss: 0.0023, MAPE: 0.0818
Epoch 251/1000, Loss: 0.0022, MAPE: 0.0759
Epoch 301/1000, Loss: 0.0022, MAPE: 0.0732
Epoch 351/1000, Loss: 0.0023, MAPE: 0.0756
Epoch 401/1000, Loss: 0.0021, MAPE: 0.0762
Epoch 451/1000, Loss: 0.0021, MAPE: 0.0799
Epoch 501/1000, Loss: 0.0022, MAPE: 0.0759
Epoch 551/1000, Loss: 0.0020, MAPE: 0.0712
Epoch 601/1000, Loss: 0.0020, MAPE: 0.0826
Epoch 651/1000, Loss: 0.0020, MAPE: 0.0811
Epoch 701/1000, Loss: 0.0020, MAPE: 0.0765
Epoch 751/1000, Loss: 0.0019, MAPE: 0.0841
Epoch 801/1000, Loss: 0.0019, MAPE: 0.0845
Epoch 851/1000, Loss: 0.0019, MAPE: 0.0709
Epoch 901/1000, Loss: 0.0018, MAPE: 0.0717
Epoch 951/1000, Loss: 0.0019, MAPE: 0.0747
